In [4]:
import pandas as pd
import json
import os
import urllib.request
import time
import warnings
warnings.filterwarnings('ignore')


# Task 1: Data Loading & Exploratory Analysis
print("--- TASK 1: Exploratory Analysis ---")
df = pd.read_csv("patient_data.csv")

print(f"Dataset Shape: {df.shape}")
print(f"Missing Values: \n{df.isnull().sum()[df.isnull().sum() > 0]}")
print("\nDescriptive Statistics (Subset):")
print(df[['age', 'days_since_last_visit', 'vitals_bp_systolic']].describe())
print("\n" + "="*50 + "\n")


# Task 2: Define Agent Tools
def get_patient_data(patient_id: str) -> str:
    """Fetch complete clinical profile for a specific patient ID."""
    patient = df[df['patient_id'] == patient_id]
    if patient.empty:
        return json.dumps({"error": f"Patient {patient_id} not found."})
    return patient.iloc[0].to_json()

def get_missed_appointments() -> str:
    """Identify patients who missed appointments and need follow-up."""
    missed = df[df['missed_last_appointment'] == 'Yes']
    cols = ['patient_id', 'patient_name', 'diagnosis', 'last_visit_date', 'notes']
    return missed[cols].to_json(orient='records')

# Define tool schemas for the LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_patient_data",
            "description": "Fetch clinical record for a single patient by ID.",
            "parameters": {
                "type": "object",
                "properties": {"patient_id": {"type": "string"}},
                "required": ["patient_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_missed_appointments",
            "description": "Fetch a list of all patients who missed their last appointment.",
            "parameters": {"type": "object", "properties": {}}
        }
    }
]


# Task 3: Implement the Agentic Loop (Gemini API Call)
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "REDACTED")
# Model endpoints with fallbacks
GEMINI_MODELS = ["gemini-3.8-flash", "gemini-3.5-flash-lite"]

def gemini_llm_call(messages):
    tools_spec = [
        {
            "functionDeclarations": [
                {
                    "name": "get_patient_data",
                    "description": "Fetch clinical record for a single patient by ID.",
                    "parameters": {
                        "type": "OBJECT",
                        "properties": {
                            "patient_id": {"type": "STRING", "description": "The patient ID"}
                        },
                        "required": ["patient_id"]
                    }
                },
                {
                    "name": "get_missed_appointments",
                    "description": "Fetch a list of all patients who missed their last appointment.",
                    "parameters": {
                        "type": "OBJECT",
                        "properties": {}
                    }
                }
            ]
        }
    ]

    contents = []
    for msg in messages:
        role = msg.get("role")
        if role == "user":
            contents.append({"role": "user", "parts": [{"text": msg["content"]}]})
        elif role == "model" or role == "assistant":
            if "part" in msg:
                contents.append({"role": "model", "parts": [msg["part"]]})
            else:
                contents.append({"role": "model", "parts": [{"text": msg["content"]}]})
        elif role == "function":
            func_response_part = {
                "functionResponse": {
                    "name": msg["name"],
                    "response": {
                        "name": msg["name"],
                        "content": json.loads(msg["content"]) if isinstance(msg["content"], str) else msg["content"]
                    }
                }
            }
            contents.append({"role": "user", "parts": [func_response_part]})

    payload = {
        "contents": contents,
        "tools": tools_spec
    }

    last_error = None
    for model in GEMINI_MODELS:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={GEMINI_API_KEY}"
        data = json.dumps(payload).encode('utf-8')
        req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})
        
        try:
            with urllib.request.urlopen(req) as resp:
                res = json.loads(resp.read().decode())
                candidate = res['candidates'][0]
                parts = candidate['content']['parts']
                for part in parts:
                    if 'functionCall' in part:
                        fn = part['functionCall']
                        return {
                            'part': part,
                            'tool_calls': [
                                {
                                    'function': {
                                        'name': fn['name'],
                                        'arguments': json.dumps(fn.get('args', {}))
                                    }
                                }
                            ]
                        }
                    elif 'text' in part:
                        return {'content': part['text']}
                return {'content': ''}
        except Exception as e:
            last_error = e
            continue

    raise Exception(f"All Gemini model attempts failed. Last error: {last_error}")


def run_agent(prompt: str):
    messages = [{"role": "user", "content": prompt}]
    
    # Step 1: Send prompt to LLM
    llm_response = gemini_llm_call(messages)
    
    # Step 2: Check for Tool Calls
    if "tool_calls" in llm_response:
        tool_call = llm_response["tool_calls"][0]
        func_name = tool_call["function"]["name"]
        args = json.loads(tool_call["function"]["arguments"])
        
        # Execute local Python function
        if func_name == "get_patient_data":
            observation = get_patient_data(**args)
        elif func_name == "get_missed_appointments":
            observation = get_missed_appointments()
            
        # Step 3: Append tool output and query LLM for final synthesis
        if "part" in llm_response:
            messages.append({"role": "model", "part": llm_response["part"]})
        messages.append({"role": "function", "name": func_name, "content": observation})
        final_response = gemini_llm_call(messages)
        return final_response["content"]
    
    return llm_response["content"]


# Task 4: Single Patient Analysis
print("--- TASK 4: Single Patient Analysis (P0003) ---")
result_t4 = run_agent("Analyze patient P0003 and flag any clinical risks.")
print(result_t4)
print("\n" + "="*50 + "\n")

# Task 5: Missed Appointment Follow-Up
print("--- TASK 5: Missed Appointment Follow-Up ---")
result_t5 = run_agent("Identify patients who missed their last appointment and generate a prioritized follow-up action plan.")
print(result_t5)


--- TASK 1: Exploratory Analysis ---
Dataset Shape: (100, 18)
Missing Values: 
Series([], dtype: int64)

Descriptive Statistics (Subset):
              age  days_since_last_visit  vitals_bp_systolic
count  100.000000             100.000000          100.000000
mean    55.360000             211.390000          145.570000
std     15.251773              87.498675           20.375889
min     28.000000              65.000000          105.000000
25%     44.000000             132.500000          130.000000
50%     55.000000             219.000000          145.500000
75%     68.000000             287.750000          161.250000
max     82.000000             360.000000          181.000000


--- TASK 4: Single Patient Analysis (P0003) ---
Based on the clinical record for **Rohan Mehta (Patient ID: P0003)**, several significant clinical risks have been flagged:

### 1. Critically Elevated Glycemic Control (Type 2 Diabetes)
* **Lab Value:** HbA1c is **11.4%**, which is severely elevated and indicate